# Color Segmentation Demo

> Demonstration on how to instantiate and execute rock cutting color segmentations utilizing the Base Color Segmentation framework.

This notebook illustrates the usage of both **Cement** and **Hydrocarbon** color models against an image input.

In [4]:
# Setup imports
import sys
import os
import cv2
import numpy as np

# Adjust path to enable subclass imports
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath("src"))

# Import Segmentor Subclasses
from src.cement_segmentation import CementSegmentation
from src.hydrocarbon_segmentation import HydrocarbonSegmentation

## 1. Prepare Target Image

Provide a target image containing geological specimens. Ensure that you have the appropriate illumination corresponding to the segmentor:
* **Cement Segmentation**: White light photography post-Phenolphthalein drop.
* **Hydrocarbon Segmentation**: UV illuminated photography.

In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import cv2
import numpy as np

# Create UI Elements
upload_btn = widgets.FileUpload(
    accept='image/*', 
    multiple=False, 
    description='Upload Image',
    button_style='info'
)

task_dropdown = widgets.Dropdown(
    options=['Cement Segmentation (Phenolphthalein)', 'Hydrocarbon Segmentation (UV)'],
    value='Cement Segmentation (Phenolphthalein)',
    description='Task:',
    layout={'width': 'max-content'}
)

process_btn = widgets.Button(
    description='Run Segmentation',
    button_style='success',
    icon='play'
)

out = widgets.Output()

def on_process_click(b):
    with out:
        clear_output()
        if not upload_btn.value:
            print("⚠️ Please upload an image first!")
            return
            
        # Robustly retrieve the uploaded file data for different ipywidgets versions
        upload_value = upload_btn.value
        try:
            if isinstance(upload_value, tuple): # ipywidgets 8.x
                content = upload_value[0].content
            else: # ipywidgets 7.x
                content = upload_value[list(upload_value.keys())[0]]['content']
        except Exception as e:
            print(f"Error extracting upload data: {e}")
            return
            
        # Decode byte stream to OpenCV RGB
        img_array = np.frombuffer(content, np.uint8)
        image_bgr = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        
        if image_bgr is None:
            print("❌ Error: Could not parse image. Please upload a valid JPG or PNG.")
            return
            
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        
        # Instantiate logic depending on the task
        if 'Cement' in task_dropdown.value:
            # Cement instances under microscope are generally larger (e.g., threshold = 500 px)
            detector = CementSegmentation(pixel_threshold=500)
            print("🔍 Running Cement Segmentation (Area > 500 px)...")
        else:
            # Hydrocarbon spots can be tiny (e.g., threshold = 10 px)
            detector = HydrocarbonSegmentation(pixel_threshold=10)
            print("🔍 Running Hydrocarbon Segmentation (Area > 10 px)...")
            
        # Execute and show visualization
        detector.visualize(image_rgb)

process_btn.on_click(on_process_click)

# Display UI
ui = widgets.VBox([
    widgets.HTML("<b>Upload Rock Sample Image for Evaluation:</b>"),
    widgets.HBox([upload_btn, task_dropdown, process_btn]),
    out
])

display(ui)